<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/05_surrogate_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 5 of 7: Surrogate Modelling with OpenImpala-Generated Data

*OpenImpala Tutorial Series — From first solve to HPC deployment*

---

A surrogate model approximates an expensive computation — in our case, a 3D PDE solve — with a fast statistical or machine learning model trained on labelled examples. The bottleneck is always the labelled data: you need many (structure, property) pairs, each requiring a full physics solve.

This tutorial uses OpenImpala to generate that dataset. We create 200 random microstructures, compute morphological features for each, solve for the ground-truth tortuosity, and then train a Random Forest regressor. Importantly, we also *explore* the data and *interpret* the model to understand which geometric features actually control transport.

**Connection to Tutorial 3 (REV):** In the [REV tutorial](03_rev_and_uncertainty.ipynb) we showed that small sub-volumes exhibit large variance in tortuosity, and that this variance reflects real structure–property correlations. Here we exploit that: each sample with its unique geometry and tortuosity becomes a training example.

**What you will learn:**
1. Generate randomised microstructures with PoreSpy and label them with OpenImpala.
2. Explore the structure–property relationships in the dataset.
3. Train a Random Forest regressor and evaluate it on held-out data.
4. Use feature importance to identify which geometric descriptors control tortuosity.

**Prerequisites:** [Tutorial 1](01_hello_openimpala.ipynb) for the OpenImpala API. [Tutorial 3](03_rev_and_uncertainty.ipynb) motivates the sampling strategy. Basic familiarity with scikit-learn is helpful.

In [ ]:
# Install OpenImpala and ML libraries
!pip install -q openimpala porespy scikit-learn matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import porespy as ps
from scipy import ndimage
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded.")

# Dataset parameters.  200 samples on 32^3 grids runs in a few minutes
# on Colab.  For a production surrogate, increase both.
N_SAMPLES = 200
SIZE = 32
np.random.seed(42)

In [ ]:
## 1. Generating the Labelled Dataset

For each sample we:
1. Generate a random structure with PoreSpy (`blobs`), varying both the target porosity and the characteristic feature size (`blobiness`).
2. Compute three morphological features that are easy to measure on any segmented image:
   - **Porosity** ($\varepsilon$) — volume fraction of the pore phase.
   - **Specific surface area** ($S_v$) — interfacial area per unit volume (units: 1/voxel).
   - **Hydraulic diameter estimate** ($d_h = 4\varepsilon / S_v$) — a classical proxy for mean pore size. It introduces a nonlinear combination of $\varepsilon$ and $S_v$ that captures how "open" the pore space is.
3. Use OpenImpala to check percolation and, if the phase is connected, solve for the ground-truth tortuosity.

In [ ]:
print("Step 1: Generating labelled data with OpenImpala...")

feature_names = ["Porosity", "Surface Area", "Hydraulic Diameter"]
X_features = []
y_target = []
n_no_percolation = 0

t_start = time.time()

with oi.Session():
    for i in range(N_SAMPLES):
        # Vary both porosity and feature size
        target_porosity = np.random.uniform(0.25, 0.75)
        blobiness = np.random.uniform(0.8, 3.0)
        micro = ps.generators.blobs(
            shape=[SIZE, SIZE, SIZE],
            porosity=target_porosity,
            blobiness=blobiness,
        ).astype(np.int32)

        # Morphological features
        porosity = oi.volume_fraction(micro, phase=1).fraction
        surface_area = ps.metrics.specific_surface_area(micro)
        hydraulic_diam = 4.0 * porosity / surface_area if surface_area > 0 else 0.0

        # Ground-truth tortuosity
        perc = oi.percolation_check(micro, phase=1, direction="z")
        if perc.percolates:
            tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity
            X_features.append([porosity, surface_area, hydraulic_diam])
            y_target.append(tau)
        else:
            n_no_percolation += 1

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{N_SAMPLES} structures processed...")

X_features = np.array(X_features)
y_target = np.array(y_target)

t_data = time.time() - t_start
print(f"\nGenerated {len(y_target)} valid samples in {t_data:.1f}s "
      f"({n_no_percolation} structures did not percolate).")

## 2. Exploring the Dataset

Before training a model, it is worth looking at the data. The scatter plots below show how each morphological feature relates to the ground-truth tortuosity. This helps us understand what the model will have to work with and whether the features carry useful information.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=120)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for ax, col, name, c in zip(axes, range(3), feature_names, colors):
    ax.scatter(X_features[:, col], y_target, s=15, alpha=0.6, color=c, edgecolor='none')
    ax.set_xlabel(name)
    ax.set_ylabel(r"Tortuosity ($\tau$)")
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Pearson correlation
    corr = np.corrcoef(X_features[:, col], y_target)[0, 1]
    ax.set_title(f"{name} (r = {corr:.2f})")

plt.suptitle(f"Feature–Tortuosity Relationships (N = {len(y_target)})", fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Training and Evaluating the Surrogate

We hold out 20% of the data for testing. A Random Forest with 100 trees is a reasonable default — it handles nonlinear relationships, requires minimal tuning, and provides built-in feature importance estimates.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Test set size:  {len(y_test)} samples")
print(f"R² score:       {r2:.3f}")
print(f"MAE:            {mae:.4f}")

# --- Evaluation plots ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

# Left: True vs. Predicted
ax1.scatter(y_test, y_pred, color='#1f77b4', s=40, alpha=0.7, edgecolor='black', linewidth=0.5)
lims = [min(y_test.min(), y_pred.min()) * 0.95, max(y_test.max(), y_pred.max()) * 1.05]
ax1.plot(lims, lims, 'r--', lw=2, label="Perfect prediction")
ax1.set_xlim(lims)
ax1.set_ylim(lims)
ax1.set_xlabel("True Tortuosity (OpenImpala)")
ax1.set_ylabel("Predicted Tortuosity (Random Forest)")
ax1.set_title(f"True vs. Predicted (R² = {r2:.2f})")
ax1.set_aspect('equal')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: Residuals
residuals = y_test - y_pred
ax2.scatter(y_test, residuals, color='#ff7f0e', s=40, alpha=0.7, edgecolor='black', linewidth=0.5)
ax2.axhline(0, color='r', linestyle='--', lw=2)
ax2.set_xlabel("True Tortuosity")
ax2.set_ylabel("Residual (True - Predicted)")
ax2.set_title(f"Residuals (MAE = {mae:.3f})")
ax2.grid(True, alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Feature Importance

Random Forest models provide a built-in measure of feature importance (mean decrease in impurity). This tells us which geometric descriptors the model relied on most heavily when predicting tortuosity. For a materials scientist, this is arguably the most interesting output: it quantifies which aspects of the microstructure actually control transport.

In [ ]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(7, 3.5), dpi=120)
ax.barh(
    [feature_names[i] for i in sorted_idx],
    importances[sorted_idx],
    color=['#1f77b4', '#ff7f0e', '#2ca02c'],
    edgecolor='black', linewidth=0.5,
)
ax.set_xlabel("Feature Importance (mean decrease in impurity)")
ax.set_title("What Controls Tortuosity?")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Print the values
for i in sorted_idx[::-1]:
    print(f"  {feature_names[i]:22s}  {importances[i]:.3f}")

## Next Steps

This tutorial demonstrated the workflow: generate structures, label them with OpenImpala, and train a surrogate. The small dataset here is a starting point — in practice you would:

- **Increase sample count** (500–5000+) for better generalisation.
- **Use richer features** — two-point correlation functions, chord-length distributions, Minkowski functionals, or learned CNN embeddings capture geometry more completely than porosity and surface area alone.
- **Use larger domains** ($64^3$–$128^3$) to reduce finite-size noise (see [Tutorial 3](03_rev_and_uncertainty.ipynb) on REV).
- **Parallelise data generation** with MPI (see [Tutorial 7](07_hpc_scaling.ipynb)) to make larger campaigns practical.

**Continue the series:**
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) — Put OpenImpala inside a design loop to optimise microstructure geometry.
- [Tutorial 7: Scaling to HPC](07_hpc_scaling.ipynb) — Generate larger datasets on a cluster with MPI.

---

## References & Further Reading

1. **OpenImpala:** S. Mayner, F. Ciucci, *OpenImpala: open-source computational framework for microstructural analysis of 3D tomography data*, SoftwareX (2024). [GitHub](https://github.com/BASE-Laboratory/OpenImpala)
2. **Surrogate models for porous media:** S. Kench & S. J. Cooper, *Generating three-dimensional structures from a two-dimensional slice with generative adversarial network-based dimensionality reduction*, Nature Machine Intelligence 3, 299–305 (2021). [doi:10.1038/s42256-021-00322-1](https://doi.org/10.1038/s42256-021-00322-1)
3. **ML for structure–property linkages:** R. Cang et al., *Microstructure representation and reconstruction of heterogeneous materials via deep belief network for computational material design*, J. Mech. Design 139(7), 071404 (2017). [doi:10.1115/1.4036649](https://doi.org/10.1115/1.4036649)
4. **Random Forests:** L. Breiman, *Random Forests*, Machine Learning 45(1), 5–32 (2001). [doi:10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)
5. **PoreSpy morphological descriptors:** J. T. Gostick et al., *PoreSpy: A Python toolkit for quantitative analysis of porous media images*, J. Open Source Software 4(37), 1296 (2019). [doi:10.21105/joss.01296](https://doi.org/10.21105/joss.01296)
6. **Feature importance interpretation:** C. Molnar, *Interpretable Machine Learning*, 2nd ed. (2022). Available at [christophm.github.io/interpretable-ml-book](https://christophm.github.io/interpretable-ml-book/)